# Image Cartoonifier

Turns a real-world photo into a cartoon-style image using classical image processing:

1. **Binary sketch** — grayscale → median blur (denoise) → Laplacian edge detection → binary threshold
2. **Color painting** — edge-preserving bilateral filter to flatten colors while keeping edges
3. **Overlay** — combine the sketch mask with the painting via `bitwise_and`

In [ ]:
import glob

import cv2
import matplotlib.pyplot as plt
import numpy as np

### Read input images

In [ ]:
image_paths = sorted(glob.glob("assets/*"))
print(image_paths)

### Pipeline

In [ ]:
def cartoonify(img, median_ksize=5, laplacian_ksize=5, threshold=160):
    """Return (binary_sketch, painting, cartoon) for a BGR input image."""
    # 1. Binary sketch: denoise -> Laplacian edges -> threshold
    grayscale = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.medianBlur(grayscale, median_ksize)
    edges = cv2.Laplacian(blurred, cv2.CV_8U, ksize=laplacian_ksize)
    _, sketch = cv2.threshold(edges, threshold, 255, cv2.THRESH_BINARY_INV)

    # 2. Painting: strong bilateral filter flattens colors, preserves edges
    painting = cv2.bilateralFilter(img, d=9, sigmaColor=250, sigmaSpace=250)

    # 3. Overlay sketch on top of the painting
    cartoon = cv2.bitwise_and(painting, painting, mask=sketch)
    return sketch, painting, cartoon

### Run on every sample image

In [ ]:
for path in image_paths:
    img = cv2.imread(path)
    if img is None:
        print("skipping unreadable file:", path)
        continue

    sketch, painting, cartoon = cartoonify(img)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, im, title in zip(
        axes,
        [cv2.cvtColor(img, cv2.COLOR_BGR2RGB), sketch, cv2.cvtColor(cartoon, cv2.COLOR_BGR2RGB)],
        ["Original", "Binary sketch", "Cartoon"],
    ):
        ax.imshow(im, cmap="gray" if im.ndim == 2 else None)
        ax.set_title(f"{title} — {path.split('/')[-1]}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()